# 🤖 DORA Report Analytical Agent

**Overview**
This notebook powers a local AI agent designed to ingest, parse, and interact with the 2025 DORA (DevOps Research and Assessment) report. By converting complex PDF layouts into LLM-friendly structures, the chatbot acts as an interactive knowledge base for DevOps practices and organizational performance.

**Architecture**
The codebase utilizes a dual-LLM structure, deploying one model as the primary analytical agent to process the report, and a second model as an independent evaluator to assess the ongoing interaction between the user and the agent.

**Core Objectives**
* **High-Fidelity Parsing:** Extracts structurally intact text from multi-column PDFs to maintain the integrity of the report's data for the local model.
* **Organizational Analysis:** Interrogates the text to extract key insights regarding leadership frameworks and organizational psychology.
* **Strategic Foresight:** Identifies critical performance indicators, pinpoints risks like the competence trap, and highlights corporate team foresight issues within modern engineering structures.

In [ ]:

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import pymupdf4llm
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
dorareport = pymupdf4llm.to_markdown("2025DORA.pdf")

In [4]:
print(dorareport)

## **State of AI-assisted Software Development** 

**2025** 

**Platinum sponsors** 

**Gold sponsors** 

**Premier research partner** 

**Research partners** 

**==> picture [282 x 35] intentionally omitted <==**

## **Contents** 

|03|65|99|
|---|---|---|
|**Executive summary**|**Platorm**|**Authors**|
||**engineering**||
|08||103|
|**Foreword**|73<br>**Value stream**<br>**management**|**Demographics and**<br>**frmographics**|
|11|||
|**Understanding your**<br>**sofware delivery**<br>**perormance**|79<br>**The AI mirror:**<br>**How AI refects**<br>**and amplifes your**|113<br>**Methodology**<br>131|
|23<br>**AI adoption and use**|**organization’s true**<br>**capabilities**|**Our research model**<br>**and its theory**|
||89||
|33|**Metrics frameworks**|137|
|**Exploring AI’s**||**Next steps**|
|**relationship to**|||
|**key outcomes**|95||
|49|**Final thoughts:**<br>**From insight**|138<br>**Appendix**|
|**DORA AI**|**to action**||
|**Capabilities Model**|||
||97||
||**Acknowledgement

In [5]:
name = "DORA expert"

In [6]:
system_prompt = f"You are acting as {name}. You are answering questions as a {name}, \
particularly questions related to a {name}'s understanding of the DORA report that talks about the state of AI assistance in software coding. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a report of {name} which you should use to answer questions. \
Be professional and engaging, as if talking to a curious reader who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## DORA Report:\n{dorareport}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [7]:
system_prompt

'You are acting as DORA expert. You are answering questions as a DORA expert, particularly questions related to a DORA expert\'s understanding of the DORA report that talks about the state of AI assistance in software coding. Your responsibility is to represent DORA expert for interactions on the website as faithfully as possible. You are given a report of DORA expert which you should use to answer questions. Be professional and engaging, as if talking to a curious reader who came across the website. If you don\'t know the answer, say so.\n\n## DORA Report:\n## **State of AI-assisted Software Development** \n\n**2025** \n\n**Platinum sponsors** \n\n**Gold sponsors** \n\n**Premier research partner** \n\n**Research partners** \n\n**==> picture [282 x 35] intentionally omitted <==**\n\n## **Contents** \n\n|03|65|99|\n|---|---|---|\n|**Executive summary**|**Platorm**|**Authors**|\n||**engineering**||\n|08||103|\n|**Foreword**|73<br>**Value stream**<br>**management**|**Demographics and**<br

In [8]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [9]:
gr.ChatInterface(chat, type="messages").launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://586dcc7063e6af7857.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [11]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their report. \
The Agent has been instructed to be professional and engaging, as if talking to a reader who came across the website. \
The Agent has been provided with context on {name} in the form of their report. Here's the information:"

evaluator_system_prompt += f"\n\n## DORA Report:\n{dorareport}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [12]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [13]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [14]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [15]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "Is DORA against climate change?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [16]:
reply

"The DORA report primarily focuses on software delivery performance and the impact of AI in software development. It does not explicitly address climate change or take a stance on environmental issues. However, organizations that adopt practices aligned with DORA's findings may find efficiencies and improvements that could contribute indirectly to sustainability efforts, such as reducing resource consumption through more efficient software practices. \n\nFor more information on DORA's perspective or initiatives related to climate change, you may want to look into specific programs or research focused on sustainability in technology. If you have further inquiries about the DORA report itself or software development practices, feel free to ask!"

In [17]:
evaluate(reply, "Is DORA against climate change?", messages[:1])

Evaluation(is_acceptable=True, feedback="The agent accurately states that the DORA report does not explicitly address climate change as a core stance. It also correctly identifies the indirect connection regarding 'reducing resource consumption through more efficient software practices', which is in line with the report's single mention of 'reducing its carbon footprint' as a reason to change an application. The tone is professional and engaging.")

In [18]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [19]:
def chat(message, history):
    if "work" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in hindi - \
              it is mandatory that you respond only and entirely in hindi"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [22]:
gr.ChatInterface(chat, type="messages").launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://81fc66ee3bcaed8cdd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
